In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.7 MoE introduction

A Mixture of Experts replaces one MLP with many, plus a **router** that sends each
token to just a few. The model gains parameters (capacity) without proportional
compute, because only the chosen experts run per token.

In [ ]:
from torch import nn

torch.manual_seed(0)
n_experts, dim = 4, 8
experts = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_experts)])
router = nn.Linear(dim, n_experts)
x = torch.randn(5, dim)
choice = router(x).argmax(-1)   # top-1 routing
out = torch.stack([experts[choice[i]](x[i]) for i in range(len(x))])
print("each token routed to one expert:", choice.tolist())
print("output shape:", tuple(out.shape))

Sparsity is the whole point: 4 experts of capacity, but each token pays for one.
Real MoEs route to the top-k and add load-balancing so experts stay busy.

In [ ]:
assert tuple(out.shape) == (5, dim)
assert int(choice.max()) < n_experts